In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split

In [2]:
data_path = "../data/raw/PS_20174392719_1491204439457_log.csv"

df = pd.read_csv(data_path)

print("Dataset loaded successfully!")
print("Shape:", df.shape)

df.head()

Dataset loaded successfully!
Shape: (6362620, 11)


,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,0
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1,0
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,0


In [3]:
columns_to_drop = [
    "nameOrig",
    "nameDest",
    "isFlaggedFraud"
]

df = df.drop(columns=columns_to_drop)

df.head()

,step,type,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,isFraud
0,1,PAYMENT,9839.64,170136.0,160296.36,0.0,0.0,0
1,1,PAYMENT,1864.28,21249.0,19384.72,0.0,0.0,0
2,1,TRANSFER,181.00,181.0,0.00,0.0,0.0,1
3,1,CASH_OUT,181.00,181.0,0.00,21182.0,0.0,1
4,1,PAYMENT,11668.14,41554.0,29885.86,0.0,0.0,0


In [4]:
df["errorBalanceOrig"] = (
    df["oldbalanceOrg"] -
    df["amount"] -
    df["newbalanceOrig"]
)

df["errorBalanceDest"] = (
    df["oldbalanceDest"] +
    df["amount"] -
    df["newbalanceDest"]
)

In [5]:
df = pd.get_dummies(
    df,
    columns=["type"],
    dtype=int
)

df.head()

,step,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,isFraud,errorBalanceOrig,errorBalanceDest,type_CASH_IN,type_CASH_OUT,type_DEBIT,type_PAYMENT,type_TRANSFER
0,1,9839.64,170136.0,160296.36,0.0,0.0,0,0.0,9839.64,0,0,0,1,0
1,1,1864.28,21249.0,19384.72,0.0,0.0,0,0.0,1864.28,0,0,0,1,0
2,1,181.00,181.0,0.00,0.0,0.0,1,0.0,181.00,0,0,0,0,1
3,1,181.00,181.0,0.00,21182.0,0.0,1,0.0,21363.00,0,1,0,0,0
4,1,11668.14,41554.0,29885.86,0.0,0.0,0,0.0,11668.14,0,0,0,1,0


In [6]:
X = df.drop(columns=["isFraud"])

y = df["isFraud"]

print("Features shape:", X.shape)
print("Target shape:", y.shape)

Features shape: (6362620, 13)
Target shape: (6362620,)


In [7]:
train_end = df["step"].quantile(0.70)
validation_end = df["step"].quantile(0.85)

train_mask = df["step"] <= train_end

validation_mask = (
    (df["step"] > train_end) &
    (df["step"] <= validation_end)
)

test_mask = df["step"] > validation_end

X_train = X.loc[train_mask]
y_train = y.loc[train_mask]

X_val = X.loc[validation_mask]
y_val = y.loc[validation_mask]

X_test = X.loc[test_mask]
y_test = y.loc[test_mask]

In [8]:
print("Training:", X_train.shape)
print("Validation:", X_val.shape)
print("Testing:", X_test.shape)

print("\nFraud distribution:")

print("Training:")
print(y_train.value_counts())

print("\nValidation:")
print(y_val.value_counts())

print("\nTesting:")
print(y_test.value_counts())

Training: (4463587, 13)
Validation: (980416, 13)
Testing: (918617, 13)

Fraud distribution:
Training:
isFraud
0    4459944
1       3643
Name: count, dtype: int64

Validation:
isFraud
0    979852
1       564
Name: count, dtype: int64

Testing:
isFraud
0    914611
1      4006
Name: count, dtype: int64
